# 06 · Analizador de trades

**Valor de un trade para un equipo:** es el cambio en los puntos esperados de su **alineación óptima** (`lineup.py`) desde ahora hasta el final de la temporada. Se calcula semana a semana, con los byes y la probabilidad de lesión, y las semanas de playoffs de la liga pesan ×2. Se evalúa **para los dos equipos** del trade.

Etapas:
- **(a)** proyecciones semana a semana y evaluación de un trade concreto ← _este notebook, por ahora_
- **(b)** incertidumbre: Monte Carlo y backtest de horizonte
- **(c)** buscador de trades 1x1 y 2x1 en los que ganan los dos equipos

Código: `src/fantasy_ml/trades.py` · configuración: `config/trades.yaml`.

## 1. Liga: calendario, fecha límite y reglas del roster

Todo se lee de la configuración de la liga en ESPN, no se escribe a mano.

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import polars as pl

from fantasy_ml import data, espn, trades as T

CFG = data.load_config("trades")
PARAMS = data.load_config("model")["params"]
GDL = ZoneInfo("America/Mexico_City")

league = espn.connect(2026)
cal = T.league_calendar(league, CFG["playoff_weight"])
rules = T.league_rules(league)
ME = espn.my_team_id()

days_left = (cal.trade_deadline - datetime.now(timezone.utc)).days
print(f"Temporada regular: semanas {cal.reg_weeks[0]}–{cal.reg_weeks[-1]} · playoffs: semanas {cal.playoff_weeks} (peso ×{cal.playoff_weight:g})")
print(f"Fecha límite de trades: {cal.trade_deadline.astimezone(GDL):%d-%m-%Y %H:%M} (Guadalajara) · faltan {days_left} días")
print(f"Semanas que se valoran: {cal.weeks[0]}–{cal.weeks[-1]} ({len(cal.weeks)})")
print(f"Slots titulares: { {k: v for k, v in rules['slots'].items() if k not in ('BE', 'IR')} }")
print(f"Roster: {CFG['max_active_roster']} activos + IR · límites por posición: {rules['position_limits']}")
if days_left < 0:
    print("⚠ La fecha límite ya pasó: los trades ya no se pueden hacer.")

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(24)

## 2. Probabilidad de jugar

El modelo predice los puntos **si el jugador juega**. En un horizonte de 15 semanas eso no basta, así que cada semana se multiplica por la probabilidad de jugar:
- **Semana actual:** según el estado de lesión de ESPN (OUT 0, doubtful 0.25, questionable 0.75).
- **Jugadores en IR:** 0 durante las próximas 4 semanas (`ir_weeks`).
- **Resto de semanas:** la **tasa histórica** de partidos jugados (2023–2025), que incluye lesiones y pérdidas de rol. Depende de la posición y del nivel de uso (`xfp`): en QB, los titulares de más uso pierden muchos menos partidos. Se interpola entre niveles para evitar saltos bruscos.

In [ ]:
src = data.load_sources()
rosters_weekly = data.load_rosters_weekly(cal.season)
state = T.season_state(src, rosters_weekly, data.load_config("scoring"), cal.season, cal.current_week)
rates = T.availability_rates(state["base"], state["points_k"], state["team_games"], CFG["availability"])
rates.select("position", "tier", "player_seasons", pl.col("xfp_center").round(1), pl.col("play_rate").round(3))

## 3. Proyección semana a semana

- **Features de forma:** congeladas en su valor actual, que es lo último que se sabe.
- **Contexto del partido de cada semana:** rival, local o visitante, descanso, estadio, y lo que permite hoy la defensa rival a esa posición.
- **Líneas de apuestas:** solo se usan las reales de la semana actual. Para las semanas siguientes se **estiman** con un modelo de fuerza ofensiva y defensiva ajustado a las líneas ya publicadas. Frente a las líneas reales de la semana 4, que ya existen, el error medio es de ~1.2 puntos de *implied total* (correlación 0.89).
- **Comprobación:** en la semana actual, esta proyección coincide exactamente con el registro de predicciones del notebook 04.

In [ ]:
proj = T.project_rest_of_season(state, PARAMS, cal, CFG)
league_proj = T.with_expected_points(proj, T.league_rosters(league), rates, cal, CFG)

missing = league_proj.filter(pl.col("proj").is_null())["name"].unique().to_list()
print(f"Jugadores con roster en la liga: {league_proj['espn_id'].n_unique()} · sin proyección: {missing or 'ninguno'}")
print(f"Filas jugador-semana: {league_proj.height:,}")

### Mi roster: puntos esperados por semana (proyección × probabilidad de jugar; `bye` = semana libre)

In [ ]:
mine = league_proj.filter(pl.col("fantasy_team_id") == ME)
wide = (mine.with_columns(cell=pl.when(pl.col("bye")).then(pl.lit("bye")).otherwise(pl.col("exp_points").round(1).cast(pl.Utf8)))
            .pivot(on="week", index=["name", "position"], values="cell", sort_columns=True))
ros = (mine.join(cal.weights(), on="week")
           .group_by("name").agg((pl.col("exp_points") * pl.col("weight")).sum().round(0).alias("ros_modelo"),
                                 (pl.col("espn_points") * pl.col("weight")).sum().round(0).alias("ros_espn"),
                                 pl.col("play_rate").first().round(2).alias("p_jugar")))
wide.join(ros, on="name").sort("ros_modelo", descending=True)

`ros_modelo` y `ros_espn` son totales ponderados (playoffs ×2) hasta la semana 17. El de ESPN es más alto por dos razones: ESPN supone que el jugador juega todos los partidos (solo descuenta byes y lesiones ya conocidas), y el modelo subestima a los titulares a inicio de temporada (notebook 03). Para valorar un trade importan las **diferencias** entre jugadores, no el nivel absoluto.

## 4. Evaluación de un trade concreto

Indica el equipo rival y los jugadores por nombre, tal como aparecen en ESPN. Para cada equipo se muestra:
- **`delta`:** cambio en el valor ponderado de su roster (playoffs ×2), desglosado en temporada regular y playoffs (sin ponderar).
- **`suelta`:** si el equipo queda con más de 15 activos, suelta al jugador que menos valor le aporta.
- **`valido`:** si el trade respeta los límites por posición.
- **Vista de ESPN:** el mismo cálculo con las proyecciones de ESPN, que es lo que ve el otro manager. Sirve para anticipar si aceptaría.

_La incertidumbre de estas cifras llega en la etapa (b)._

In [ ]:
TRADE = {"partner": "David", "give": ["Brock Purdy"], "get": ["Nico Collins"]}

partner = T.resolve_team(league_proj, TRADE["partner"])
give = T.resolve_players(league_proj, ME, TRADE["give"])
get = T.resolve_players(league_proj, partner, TRADE["get"])

res_model = T.evaluate_trade(league_proj, ME, partner, give, get, rules, cal, CFG, score="exp_points")
res_espn = T.evaluate_trade(league_proj, ME, partner, give, get, rules, cal, CFG, score="espn_points")
(res_model.select("fantasy_team", "da", "recibe", "suelta", "valor_antes", "valor_despues", "delta", "delta_regular", "delta_playoffs", "valido", "motivo")
          .join(res_espn.select("fantasy_team", delta_segun_espn="delta"), on="fantasy_team"))

In [ ]:
both_win = (res_model["delta"] > 0).all()
espn_other = res_espn.filter(pl.col("fantasy_team_id") == partner)["delta"][0]
print("Según el modelo:", "ganan los dos equipos ✓" if both_win else "no ganan los dos ✗")
print(f"Según ESPN, {TRADE['partner']} {'gana' if espn_other > 0 else 'pierde'} {abs(espn_other):.1f} puntos ponderados "
      f"→ {'probablemente lo aceptaría' if espn_other > 0 else 'probablemente lo rechazaría si decide mirando a ESPN'}")

### ¿De dónde sale el valor? Desglose semana a semana (alineación óptima antes y después)

In [ ]:
pl.concat([
    T.weekly_trade_breakdown(league_proj, ME, partner, give, get, rules, cal, CFG).with_columns(equipo=pl.lit("yo")),
    T.weekly_trade_breakdown(league_proj, partner, ME, get, give, rules, cal, CFG).with_columns(equipo=pl.lit(TRADE["partner"])),
]).pivot(on="equipo", index=["week", "phase", "weight"], values="delta").with_columns(pl.exclude("week", "phase", "weight").round(1))